# train_upload_hive (simplified)

A slimmed-down `train_upload_hive_EN.ipynb`: the **Hive connection, data fetch, and dependency installation** logic lives in `train_upload_hive_lib_EN.py` (upload it to ModelArts in the **same directory** as this notebook); the notebook keeps only configuration and the train/upload flow. For the full step-by-step troubleshooting edition see `train_upload_hive_EN.ipynb`.

## 1. Configuration ⚠️ Must Edit

> Change `OBS_BUCKET` below to your own bucket name; when switching regions, update `OBS_ENDPOINT` to match.
> If the Notebook is bound to an OBS agency, leave AK/SK empty (moxing then uses the agency automatically);
> otherwise fill in your own AK/SK temporarily — **clear them as soon as you are done, and never commit real keys to the repository or share them**.


In [ ]:
# ==================== ⚠️ MUST EDIT ====================
OBS_BUCKET   = "<your-bucket-name>"                    # Your OBS bucket name
OBS_PREFIX   = "models"                                # OBS storage path prefix (= the first half of the server-side OBS_KEY)
OBS_ENDPOINT = "obs.cn-north-4.myhuaweicloud.com"      # OBS endpoint (update when switching regions; the deployment-side OBS_ENDPOINT must match)

# IAM access keys (Huawei Cloud console → My Credentials → Access Keys)
# ⚠️ Leave empty if the Notebook is bound to an OBS agency (moxing uses the agency automatically); fill in your own only when there is no agency
# ⚠️ Clear real AK/SK as soon as you are done — never commit them to the repository or share them
ACCESS_KEY_ID     = ""    # ← fill in your AK (or leave empty to use the agency)
SECRET_ACCESS_KEY = ""    # ← fill in your SK (or leave empty to use the agency)
# ==========================================================

import os
from pathlib import Path

# Working directory: a ModelArts Notebook uses its built-in writable directory; a local run falls back to the current directory
WORK_DIR = "/home/ma-user/work/xgb_train" if Path("/home/ma-user").exists() else "."

assert "<" not in OBS_BUCKET, "Please replace <your-bucket-name> with the actual bucket name first"

WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
OLD_DIR = WORK / "model_out" / "old"
NEW_DIR = WORK / "model_out" / "new"
OLD_DIR.mkdir(parents=True, exist_ok=True)
NEW_DIR.mkdir(parents=True, exist_ok=True)

# Keep both model sets locally (for comparison verification)
OLD_MODEL_LOCAL = OLD_DIR / "xgboost_breast_cancer.json"
NEW_MODEL_LOCAL = NEW_DIR / "xgboost_breast_cancer.json"

# OBS has a single target path (no old/new subdirectories; switching happens via ACTIVE_MODEL in §7)
ACTIVE_MODEL_OBS = f"obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json"

print(f"Working directory: {WORK}")
print(f"OBS bucket:        {OBS_BUCKET}")
print(f"OBS target path:   {ACTIVE_MODEL_OBS}")

# --- MRS Hive connection settings (measured values; source of truth: hive_export/MRS_RUN.md §0) ---
HIVE_HOST = "10.0.0.15"    # HiveServer2 internal IP (master1)
HIVE_PORT = 21066          # HiveServer2 Thrift port
DATABASE  = "default"
USERNAME  = "hhx"          # MRS business user (kinit prompts for its password)
REALM     = "252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM"
KDC_HOSTS = ["10.0.0.15", "10.0.0.51"]  # both masters, for failover
KDC_PORT  = 21732          # Huawei MRS KDC port, NOT 88!
print(f"Hive: {HIVE_HOST}:{HIVE_PORT}  user: {USERNAME}@{REALM}")


## 2. Prepare Dependencies

`ensure_training_deps()` from `train_upload_hive_lib_EN.py` installs whatever is missing (pandas / scikit-learn / xgboost) with a **heartbeat + live progress**; already-installed packages are skipped. The ModelArts pandas ABI fix (removing modelarts-sdk from sys.path) happens automatically the moment you run `from ... import *`.


In [ ]:
from train_upload_hive_lib_EN import *

ensure_training_deps()   # install what is missing (heartbeat + progress); usually preinstalled on ModelArts

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

print(f"pandas {pd.__version__} <- {pd.__file__}")
print(f"numpy  {np.__version__}")

# ModelArts' bundled OBS library (moxing auth depends on §1 AK/SK, so it stays in the notebook)
try:
    import moxing as mox
    if ACCESS_KEY_ID and SECRET_ACCESS_KEY:
        import moxing.framework.content_db as content_db
        content_db.configure_obs_credentials(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            endpoint=OBS_ENDPOINT,
        )
        print("moxing auth configured with AK/SK")
    else:
        print("no AK/SK provided, using the Notebook agency")
    HAS_MOXING = True
except ImportError:
    HAS_MOXING = False
    print("moxing unavailable, will fall back to esdk-obs-python")


## 3. Connect to MRS Hive (Kerberos-secured cluster)

One call does it all: network probe → environment setup (kinit/cyrus-sasl/pyhive, **with heartbeat progress**) → krb5.conf → kinit (a password prompt for `hhx`) → connect to HiveServer2. Defaults are the measured values; step-by-step troubleshooting and rationale in `modelarts_hive_conn_EN.ipynb`, `MRS_RUN.md`, ADR-0002.

> Fully idempotent: **after ticket expiry or a kernel restart, just re-run this cell** (installed deps and valid tickets are skipped automatically).


In [ ]:
conn = connect_mrs_hive(
    hive_host=HIVE_HOST, hive_port=HIVE_PORT, database=DATABASE,
    username=USERNAME, realm=REALM, kdc_hosts=KDC_HOSTS, kdc_port=KDC_PORT,
)


## 4. Load Data from Hive + Define the Sample

`fetch_breast_cancer(conn)` (in the lib) fetches in small batches of 5 rows to dodge the large-frame decode bug in some libsasl2 builds. This cell restores sklearn's spaced feature names from Hive's underscored columns, validates the data, and embeds the test sample from `sample_request.json`.


In [ ]:
# === Read from Hive (small-batch fetch lives in the lib, dodging the libsasl2 large-frame bug) ===
df_hive = fetch_breast_cancer(conn)

# Hive columns use underscores (mean_radius); restore sklearn's spaced style
# ("mean radius") so feature names match sample_request.json / app.py
df_hive.columns = [c.replace("_", " ") for c in df_hive.columns]

X = df_hive.drop(columns=["target"])
y = df_hive["target"].astype(int)
print(f"Hive breast_cancer: {X.shape[0]} samples, {X.shape[1]} features")

# === Data validation ===
# Feature names are borrowed from sklearn only (not its data) to guarantee the
# same feature order as the inference service
from sklearn.datasets import load_breast_cancer
FEATURE_NAMES = list(load_breast_cancer().feature_names)
assert list(X.columns) == FEATURE_NAMES, f"columns mismatch sklearn: {list(X.columns)[:3]} ..."
assert X.shape == (569, 30), f"expected 569x30, got {X.shape}"
assert set(y.unique()) <= {0, 1}, f"unexpected target values: {sorted(y.unique())}"
print("[OK] validation passed: 569x30, names match sklearn, target in {0,1}")

# === Embed the test sample (from sample_request.json) ===
sample_row = {
    "mean radius": 17.99, "mean texture": 10.38, "mean perimeter": 122.8,
    "mean area": 1001.0, "mean smoothness": 0.1184, "mean compactness": 0.2776,
    "mean concavity": 0.3001, "mean concave points": 0.1471,
    "mean symmetry": 0.2419, "mean fractal dimension": 0.07871,
    "radius error": 1.095, "texture error": 0.9053, "perimeter error": 8.589,
    "area error": 153.4, "smoothness error": 0.006399,
    "compactness error": 0.04904, "concavity error": 0.05373,
    "concave points error": 0.01587, "symmetry error": 0.03003,
    "fractal dimension error": 0.006193, "worst radius": 25.38,
    "worst texture": 17.33, "worst perimeter": 184.6, "worst area": 2019.0,
    "worst smoothness": 0.1622, "worst compactness": 0.6656,
    "worst concavity": 0.7119, "worst concave points": 0.2654,
    "worst symmetry": 0.4601, "worst fractal dimension": 0.1189,
}
sample_df = pd.DataFrame([sample_row], columns=FEATURE_NAMES)
print(f"test sample: {len(sample_row)} features")


## 5. Training Function

Train → evaluate → save locally → print the sample's predicted value.
The old and new models give different predictions for the same sample — that difference
is exactly the criterion for the hot swap verification later on.


In [ ]:
def train_and_save(params, random_state, output_path, label):
    """Train a model, evaluate and save it, and return the sample's prediction probability."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y,
    )
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=random_state,
        **params,
    )
    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    pred = float(model.predict_proba(sample_df)[0, 1])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(output_path))
    size = output_path.stat().st_size

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Hyperparams: {params}")
    print(f"  Accuracy:    {acc:.4f}  AUC: {auc:.4f}")
    print(f"  Sample prediction: {pred:.16f}")
    print(f"  Saved locally:     {output_path} ({size:,} bytes)")
    return pred


## 6. Train the OLD Model (baseline)

100 trees, shallow depth, high learning rate.


In [ ]:
old_pred = train_and_save(
    params=dict(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
    ),
    random_state=42,
    output_path=OLD_MODEL_LOCAL,
    label="OLD MODEL (baseline)",
)


## 7. Train the NEW Model (different hyperparameters)

250 trees, deeper trees, low learning rate, with regularization.


In [ ]:
new_pred = train_and_save(
    params=dict(
        n_estimators=250, max_depth=6, learning_rate=0.01,
        subsample=0.6, colsample_bytree=0.5,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0, gamma=0.5,
    ),
    random_state=2024,
    output_path=NEW_MODEL_LOCAL,
    label="NEW MODEL (updated)",
)


## 8. Upload the Model to OBS 🚀

Upload the trained model to the **single target path** on OBS (the inference service
only ever reads from this path):

- `obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json`

> The inference service (`app.py`, environment variables `OBS_BUCKET` + `OBS_KEY`) reads
> the model from this path.
> To switch between old / new: change `ACTIVE_MODEL` below and re-run this cell.


In [ ]:
# Choose which model set to upload to OBS (set to "new" to switch to the new model)
ACTIVE_MODEL = "old"   # "old" or "new"

LOCAL_TO_UPLOAD = OLD_MODEL_LOCAL if ACTIVE_MODEL == "old" else NEW_MODEL_LOCAL
print(f"Current selection: {ACTIVE_MODEL} model")
print(f"Local file:        {LOCAL_TO_UPLOAD} ({LOCAL_TO_UPLOAD.stat().st_size:,} bytes)")
print(f"OBS target:        {ACTIVE_MODEL_OBS}")
print()

def upload_to_obs(local_path, obs_uri, label=""):
    """Upload a single file to OBS (overwrites)."""
    tag = f" [{label}]" if label else ""
    print(f"  Upload{tag}: {local_path} → {obs_uri}")

    if HAS_MOXING:
        mox.file.copy(str(local_path), obs_uri)
    else:
        # fallback: esdk-obs-python
        from obs import ObsClient
        assert ACCESS_KEY_ID and SECRET_ACCESS_KEY, "AK/SK are required when moxing is unavailable"
        client = ObsClient(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            server=f"https://{OBS_ENDPOINT}",
        )
        key = obs_uri.replace(f"obs://{OBS_BUCKET}/", "")
        resp = client.putFile(OBS_BUCKET, key, str(local_path))
        assert resp.status < 300, f"Upload failed: status={resp.status}"
        client.close()

    size = local_path.stat().st_size
    print(f"    ✅ Done ({size:,} bytes)")

upload_to_obs(LOCAL_TO_UPLOAD, ACTIVE_MODEL_OBS, ACTIVE_MODEL.upper())
print(f"\n🚀 Upload complete!")
print(f"   The inference service app.py reads the model from {ACTIVE_MODEL_OBS}.")
print(f"   To switch models: set ACTIVE_MODEL to 'new' and re-run this cell.")
